# Fine Tune Gemma 4 E2B

**Parameter Efficient Fine Tuning** (_PEFT_) of [Gemma 4 E2B](https://huggingface.co/google/gemma-4-E2B-it) model on a custom *Email Dataset* using *Low-Rank Adaptation (LoRA)*

**[Unsloth](https://unsloth.ai)** has been used over the default *transformers* library because they provide an optimized version of [Gemma 4 E2B](https://huggingface.co/unsloth/gemma-4-E2B-it-unsloth-bnb-4bit) which enables fine-tuning on low VRAM GPUs like NVIDIA T4 from Google Colab Free Account

The code has been borrowed from the [official Notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks) from Unsloth.

## Install Dependencies

In [ ]:
# ! pip3 install --no-cache-dir --progress-bar=off "unsloth" >> install.log
# ! pip3 install --no-cache-dir --progress-bar=off --no-deps transformers==5.5.0 >> install.log
# ! pip3 install --no-cache-dir --progress-bar=off --no-deps torchcodec >> install.log
# ! pip3 install --no-cache-dir --progress-bar=off --no-deps timm >> install.log

## Setup Model

A **Huggingface Access Token** is needed to fetch the dataset from Huggingface and also backup the fine-tuned model after training

Getting Token:

1. Login to Huggingface
2. Profile
3. Access Tokens
4. Create Access Token
5. Token type to Write
6. Create Token
7. Paste token in Notebook

In [ ]:
# @title Login to Huggingface {"run":"auto","form-width":"50%","single-column":true}
HF_TOKEN = "" # @param {"type":"string"}

import huggingface_hub
huggingface_hub.login(token=HF_TOKEN)

## Model

We will be using 4-bit [Gemma 4 E2B](https://huggingface.co/unsloth/gemma-4-E2B-it-unsloth-bnb-4bit) model by unsloth with a context lenth of 1536 for maximum memory efficiency while retaining the capacity to learn from long emails.

Since base model has vision capabilities, our objective does not involve images so we will not fine tune the vision layers and only tune the Dense and Attention layers.

### Low-Rank Adaptation

LoRA (Low-Rank Adaptation) fine-tunes large models by keeping pre-trained weights frozen and injecting trainable, low-rank matrices into each layer to drastically reduce the number of parameters being updated. We can think of it like as approximating a full size weight matrix with the product of two lower rank matrices.

* **Rank:** Determines the dimensionality of the low-rank matrices, balances the degree of model's learning capacity against the number of trainable parameters.

* **Scale ($\alpha$):** A constant factor used to adjust the influence of the LoRA weights relative to the frozen pre-trained weights.

* **Dropout:** A regularization technique that randomly zeros out elements of the LoRA adapters during training to prevent overfitting.

* **Bias:** An optional parameter that allows for the fine-tuning of the layer's additive offsets alongside the low-rank weight updates.

In [ ]:
# @title Load Model & Tokenizer {"run":"auto","single-column":true}
MODEL_REPO_ID = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit" # @param {"type":"string"}
LOAD_4_BIT = True # @param {"type":"boolean","placeholder":"True"}
LOAD_8_BIT = False # @param {"type":"boolean","placeholder":"False"}
FULL_FINETUNING = False # @param {"type":"boolean","placeholder":"False"}
CONTEXT_LENGTH = 1536 # @param {"type":"slider","min":512,"max":4096,"step":32}


from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    MODEL_REPO_ID,
    dtype=None,
    load_in_4bit=LOAD_4_BIT,
    load_in_8bit=LOAD_8_BIT,
    max_seq_length=CONTEXT_LENGTH,
    full_finetuning=FULL_FINETUNING
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [ ]:
# @title Config PEFT LoRA parameters {"run":"auto","single-column":true}
RANK = 4 # @param ["1","2","4","8","16","32"] {"type":"raw"}
SCALE = 4 # @param {"type":"slider","min":1,"max":32,"step":1}
LORA_DROPOUT = 0 # @param {"type":"slider","min":0,"max":0.5,"step":0.05}
TUNE_VISION_LAYERS = False # @param {"type":"boolean","placeholder":"False"}
TUNE_LANGUAGE_LAYERS = True # @param {"type":"boolean","placeholder":"True"}
TUNE_ATTENTION_LAYERS = True # @param {"type":"boolean","placeholder":"False"}
TUNE_DENSE_LAYERS = True # @param {"type":"boolean","placeholder":"True"}




model = FastModel.get_peft_model(
    model = model,

    # LoRA Config
    r = RANK,
    lora_alpha = SCALE,
    lora_dropout = LORA_DROPOUT,
    bias = "none",

    # Layers
    finetune_vision_layers = TUNE_VISION_LAYERS,
    finetune_language_layers = TUNE_LANGUAGE_LAYERS,
    finetune_attention_layers = TUNE_ATTENTION_LAYERS,
    finetune_mlp_layers = TUNE_DENSE_LAYERS
)

## Data Pipeline

The data pipeline takes input our custom dataset and transforms it into a simple text form which the model can parse with ease.

In [ ]:
# @title Define Pipeline
SYSTEM_PROMPT = "You are an automated Sales email agent." # @param {"type":"string"}


import json

def convert_to_prompt_response(example: dict):
  return {
      "prompt": example["prompt"],
      "response": {k: v for k, v in example.items() if k != "prompt"}
  }


def convert_to_chatml(example: dict):
  return {
      "messages": [
          {"role": "system", "content": SYSTEM_PROMPT},
          {"role": "user", "content": example["prompt"]},
          {"role": "assistant", "content": json.dumps(example["response"])}
      ]
  }


def apply_chat_template(example: dict):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        ).removeprefix("<bos>")
    }


In [ ]:
# @title Load & Process Dataset
DATASET_REPO_ID = "pitangent/email-training-dataset" # @param {"type":"string"}


from datasets.load import load_dataset

DATASET = (
    load_dataset(DATASET_REPO_ID, split="train")
    .map(convert_to_prompt_response)
    .select_columns(["prompt", "response"])
    .map(convert_to_chatml)
    .select_columns(["messages"])
    .map(apply_chat_template)
    .select_columns(["text"])
)

## Setup Trainer

In [ ]:
# @title Tuning Hyper-Parameters {"run":"auto","single-column":true}
OUTPUT_DIRECTORY = "Gemma-4-E2B-it-Emails-LoRA" # @param {"type":"string"}
OPTIMIZER = "adamw_8bit" # @param ["adamw_8bit"]
LEARNING_RATE = 0.00005 # @param {"type":"slider","min":0.000001,"max":0.0001,"step":0.000001}
LEARNING_RATE_SCHEDULER = "cosine" # @param ["cosine","linear"]
WARMUP_STEPS = 5 # @param {"type":"integer","placeholder":"3"}
MAX_STEPS = 100 # @param {"type":"integer","placeholder":"100"}
LOGGING_STEPS = 5 # @param {"type":"integer","placeholder":"5"}
CHECKPOINT_STEPS = 50 # @param {"type":"integer","placeholder":"50"}
PUSH_TO_HUB = True # @param {"type":"boolean"}
TRAINING_BATCH = 1 # @param {"type":"integer","placeholder":"1"}
GRADIENT_ACCUMULATION_STEPS = 1 # @param {"type":"integer","placeholder":"1"}
SEED = 49605 # @param {"type":"slider","min":1000,"max":100000,"step":1}



from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    output_dir = OUTPUT_DIRECTORY,

    # optimization
    optim = OPTIMIZER,
    learning_rate = LEARNING_RATE,
    lr_scheduler_type = LEARNING_RATE_SCHEDULER,
    weight_decay = 1e-4,

    # steps
    warmup_steps = WARMUP_STEPS,
    max_steps = MAX_STEPS,
    logging_steps = LOGGING_STEPS,
    save_steps = CHECKPOINT_STEPS,
    push_to_hub = PUSH_TO_HUB,
    report_to = "tensorboard",

    # batching
    per_device_train_batch_size = TRAINING_BATCH,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,

    # others
    seed = SEED
)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = DATASET,
    eval_dataset = None,
    args = config
)

**Targeted Learning**: By default the model trains on the entire data which means that it train on user's messages which is not needed. We only want the model to generate responses based on the user's message.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

**Start Training**

In [ ]:
stats = trainer.train()